# 06 — Train/Test Split Strategy: Leave-One-Cycle-Out

**Goal:** Produce the leave-one-cycle-out cross-validation groupings that notebooks 07–08 will
loop through. This notebook's *only* job is generating correct splits — no training, no
evaluation here.

**Why not a random row split?** Hourly rows within a cycle are highly correlated (hour 14 and
hour 15 of the same cycle are nearly identical). Randomly shuffling rows would leak near-duplicate
information across train/test and inflate accuracy. Splitting must happen at the **cycle** level,
done separately per condition (indoor / outdoor evaluated independently).

In [1]:
import pandas as pd
import json
from pathlib import Path

PREPROCESSED_DIR = Path("../data/preprocessed")
df = pd.read_csv(PREPROCESSED_DIR / "labeled_dataset.csv")

FEATURE_COLS = ["temperature_C", "humidity_pct", "soil_moisture_pct", "moisture_trend", "light_lux", "hour_of_day"]
LABEL_COL = "urgency_class"

print(df.shape)
df["cycle_id"].unique()

(82, 11)


<StringArray>
['indoor_cycle01', 'indoor_cycle02', 'outdoor_cycle01', 'outdoor_cycle02']
Length: 4, dtype: str

## Build leave-one-cycle-out folds, per condition

In [2]:
def build_loco_folds(df, condition):
    subset = df[df["condition"] == condition]
    cycle_ids = subset["cycle_id"].unique()
    folds = []
    for test_cycle in cycle_ids:
        train_cycles = [c for c in cycle_ids if c != test_cycle]
        folds.append({"test_cycle": test_cycle, "train_cycles": list(train_cycles)})
    return folds

indoor_folds = build_loco_folds(df, "indoor")
outdoor_folds = build_loco_folds(df, "outdoor")

print(f"Indoor: {len(indoor_folds)} folds (one per cycle)")
for f in indoor_folds:
    print(f"  test={f['test_cycle']:20s} train={f['train_cycles']}")

print(f"\nOutdoor: {len(outdoor_folds)} folds")
for f in outdoor_folds:
    print(f"  test={f['test_cycle']:20s} train={f['train_cycles']}")

Indoor: 2 folds (one per cycle)
  test=indoor_cycle01       train=['indoor_cycle02']
  test=indoor_cycle02       train=['indoor_cycle01']

Outdoor: 2 folds
  test=outdoor_cycle01      train=['outdoor_cycle02']
  test=outdoor_cycle02      train=['outdoor_cycle01']


## Verify no data leakage

Every row from a given cycle must end up entirely in train OR entirely in test for that fold — never split across both.

In [3]:
def verify_no_leakage(df, folds, condition):
    subset = df[df["condition"] == condition]
    for fold in folds:
        test_rows = subset[subset["cycle_id"] == fold["test_cycle"]]
        train_rows = subset[subset["cycle_id"].isin(fold["train_cycles"])]
        overlap = set(test_rows["cycle_id"]) & set(train_rows["cycle_id"])
        assert len(overlap) == 0, f"LEAKAGE in fold testing {fold['test_cycle']}: {overlap}"
    print(f"{condition}: no leakage across {len(folds)} folds — verified.")

verify_no_leakage(df, indoor_folds, "indoor")
verify_no_leakage(df, outdoor_folds, "outdoor")

indoor: no leakage across 2 folds — verified.
outdoor: no leakage across 2 folds — verified.


## Helper: materialize a fold into actual train/test DataFrames

Used by notebooks 07–08.

In [4]:
def get_fold_data(df, condition, fold, feature_cols=FEATURE_COLS, label_col=LABEL_COL):
    subset = df[df["condition"] == condition]
    train = subset[subset["cycle_id"].isin(fold["train_cycles"])]
    test = subset[subset["cycle_id"] == fold["test_cycle"]]
    X_train, y_train = train[feature_cols], train[label_col]
    X_test, y_test = test[feature_cols], test[label_col]
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = get_fold_data(df, "indoor", indoor_folds[0])
print(f"Fold testing {indoor_folds[0]['test_cycle']}: train={len(X_train)} rows, test={len(X_test)} rows")

Fold testing indoor_cycle01: train=25 rows, test=25 rows


## Save fold definitions for reuse

In [5]:
folds_export = {"indoor": indoor_folds, "outdoor": outdoor_folds}
with open(PREPROCESSED_DIR / "loco_folds.json", "w") as f:
    json.dump(folds_export, f, indent=2)

print("Saved loco_folds.json")

Saved loco_folds.json


**Next step:** `07_model_training.ipynb` — train logistic regression, decision tree, and random forest across every fold.